In [1]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path.cwd().parents[0]))

%load_ext autoreload
%autoreload 2


In [2]:
from alz_prog_net.model import AlzProgNet
import keras

model = AlzProgNet(
    num_modalities=5,
    modalities_hidden_dims=[32, 256],
    modality_output_dim=128,
    latent_dim=512,
    num_transformer_layers=3,
    num_heads=4,
    ff_dim=512,
    pooling_hidden_dim=32,
    modality_dropout=0.1,
    # progression_hidden_dims=(256, 128, 32),
    progression_hidden_dims=(256, 128, 16),
    time_points=(0, 6, 12, 24),
    temporal_levels=(True, True, False),
    time_dim=16,
    n_time_frequencies=4,
    use_time=True,
    use_gate=True,
    use_residual=True,
    use_interaction=True,

    initial_residual_scale=0.25,
    delta_dropout=0.05,
    output_dim=3
)

input_dims = [6, 2, 11, 3, 4]
inputs = [
    keras.Input(shape=(dim,), name=f"modality_{i}")
    for i, dim in enumerate(input_dims)
]

outputs = model(inputs)



I0000 00:00:1791496919.573489   30066 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791496919.624999   30066 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791496920.459371   30066 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791496921.991195   30066 gpu_device.cc:2043] Creat

In [3]:
model.summary(expand_nested=True)

Model: "alz_prog_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ modality_encoder                │ ?                      │        41,568 │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_1              │ ?                      │        41,440 │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_2              │ ?                      │        41,728 │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_3              │ ?                      │        41,472 │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ modality_encoder_4              │ ?                      │        41,504 │
│ (ModalityEncoder)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_0                   │ ?                      │       262,400 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_1                   │ ?                      │       262,400 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_2                   │ ?                      │       262,400 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_pooling               │ ?                      │         4,160 │
│ (AttentionPooling)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_norm                     │ (None, 128)            │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ swi_glu (SwiGLU)                │ ?                      │       132,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ disease_progression             │ ?                      │     2,081,607 │
│ (DiseaseProgressionDecoder)     │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,213,031 (12.26 MB)

 Trainable params: 3,213,031 (12.26 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
for encoder, inp in zip(model.modality_encoders, inputs):
    y = encoder(inp)
    encoder.trainable_weights
    print("encoder:", y.shape, encoder.name)

encoder: (None, 1, 128) modality_encoder_5
encoder: (None, 1, 128) modality_encoder_6
encoder: (None, 1, 128) modality_encoder_7
encoder: (None, 1, 128) modality_encoder_8
encoder: (None, 1, 128) modality_encoder_9
